# Pagrindinis metodas: CatBoost

**Uždavinys:** $\hat p(y=1\mid x)$ — tikimybė, kad klientas sutiks su terminuotu indėliu.

CatBoost kuria **medžių ansamblį**. Formulė dokumente:

$$F_M(x)=\sum_{m=1}^{M}\gamma_m h_m(x),\qquad \hat p=\sigma(F_M(x))$$

$h_m$ — m-tojo medžio nuosprendis, $\gamma_m$ — mokymosi žingsnis (`learning_rate`), $M$ — `iterations`, $\sigma$ — sigmoidė (`predict_proba`).

Šis notebook savarankiškas Colab (įkelkite `bank-full.csv`). Reikia: `pip install catboost`.

## 0. Bibliotekos

CatBoost Colab'e paprastai nėra iš karto — įdiegiame fiksuotą paketą.

In [ ]:
try:
    from catboost import CatBoostClassifier
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "catboost"])
    from catboost import CatBoostClassifier
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import average_precision_score, brier_score_loss, precision_recall_curve
from sklearn.calibration import calibration_curve

from sklearn.linear_model import LogisticRegression
from IPython.display import display


## 1. Bendros taisyklės

Ta pati sėkla ir tas pats chronologinis skaidymas kaip kituose notebook'uose, kad modelius būtų galima lyginti.

In [ ]:
# Fiksuota sėkla visur — paleidus du kartus rezultatai turi sutapti.
SEED = 42
C_CALL = 1.0      # sąlyginis vieno skambučio kaštas
V_SUCCESS = 10.0  # sąlyginė sėkmingo indėlio vertė (parametrai derinami su banku)
K_FRACTION = 0.10 # precision@k: k = 10 % test imties (ribotas operatorių biudžetas)

import os, random
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

DATA_PATH = "bank-full.csv"
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CAT_COLS = ["job","marital","education","default","housing","loan","contact","month","poutcome"]
NUM_BASE = ["age","balance","day","campaign","pdays","previous","never_contacted"]

def load_bank(path=None):
    path = path or DATA_PATH
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Nerastas {path}. Colab: kairėje Files (aplankas) → Upload → įkelkite bank-full.csv "
            "į tą pačią sesiją kaip šis notebook."
        )
    df = pd.read_csv(path, sep=";")
    print(f"Nuskaityta: {df.shape[0]} eil. × {df.shape[1]} stulp.")
    return df

def month_change_indices(df):
    m = df["month"].to_numpy()
    ch = [0]
    for i in range(1, len(m)):
        if m[i] != m[i-1]:
            ch.append(i)
    ch.append(len(df))
    return ch

def chronological_split(df):
    """~70/15/15 pagal eilutės tvarką, ribos prie mėnesio virsmo."""
    n = len(df)
    ch = month_change_indices(df)
    def nearest(t):
        return min(ch, key=lambda c: abs(c-t))
    train_end = nearest(int(0.70 * n))
    val_end = nearest(int(0.85 * n))
    if train_end >= val_end:
        val_end = min(n, train_end + int(0.15 * n))
    train, val, test = df.iloc[:train_end].copy(), df.iloc[train_end:val_end].copy(), df.iloc[val_end:].copy()
    print(f"Skaidymas train/val/test: {len(train)}/{len(val)}/{len(test)}  ribos={train_end},{val_end}")
    print(f"  train month nuo {train['month'].iloc[0]} iki {train['month'].iloc[-1]}")
    print(f"  val   month nuo {val['month'].iloc[0]} iki {val['month'].iloc[-1]}")
    print(f"  test  month nuo {test['month'].iloc[0]} iki {test['month'].iloc[-1]}")
    return train, val, test

def temporal_shift_split(df):
    """
    Tas pats mokymas (pirmos ~70 %). Du testai:
    arti = vidurys (~15 %), toli = pabaiga (~15 %).
    """
    n = len(df)
    ch = month_change_indices(df)
    def nearest(t):
        return min(ch, key=lambda c: abs(c-t))
    train_end = nearest(int(0.70 * n))
    val_end = nearest(int(0.85 * n))
    if train_end >= val_end:
        val_end = min(n, train_end + int(0.15 * n))
    train = df.iloc[:train_end].copy()
    near = df.iloc[train_end:val_end].copy()
    far = df.iloc[val_end:].copy()
    print(
        f"Laiko poslinkis: mokymas n={len(train)}; "
        f"arti (vidurys) n={len(near)}; toli (pabaiga) n={len(far)}"
    )
    return train, near, far

def add_features(df):
    out = df.copy()
    # pdays=-1: klientas niekada nekontaktuotas anksčiau → atskiras požymis,
    # o -1 pakeičiame 0, kad tai nebūtų „neigiama trukmė“.
    out["never_contacted"] = (out["pdays"] == -1).astype(int)
    out.loc[out["pdays"] == -1, "pdays"] = 0
    out["y_bin"] = (out["y"].astype(str).str.lower() == "yes").astype(int)
    return out

def cap_previous(train, *others):
    cap = float(train["previous"].quantile(0.99))
    print(f"previous apkirpimas ties train 99-uoju procentiliu = {cap:.2f} (max buvo {train['previous'].max()})")
    out = []
    for p in (train,) + others:
        q = p.copy()
        q["previous"] = q["previous"].clip(upper=cap)
        out.append(q)
    return tuple(out)

def one_hot():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def make_preprocessor(include_duration):
    num = NUM_BASE + (["duration"] if include_duration else [])
    return ColumnTransformer([
        ("num", StandardScaler(), num),
        ("cat", one_hot(), CAT_COLS),
    ], remainder="drop")

def Xy(df, include_duration):
    cols = CAT_COLS + NUM_BASE + (["duration"] if include_duration else [])
    return df[cols].copy(), df["y_bin"].to_numpy()

def precision_at_k(y, p, k):
    k = int(min(k, len(y)))
    order = np.argsort(-np.asarray(p), kind="mergesort")
    return float(np.asarray(y)[order][:k].mean())

def contact_cost(y, yhat, c_call=C_CALL, v_success=V_SUCCESS):
    y = np.asarray(y).astype(int); yhat = np.asarray(yhat).astype(int)
    tp = int(((yhat==1)&(y==1)).sum()); fp = int(((yhat==1)&(y==0)).sum())
    return float(c_call*(tp+fp) - v_success*tp)

def best_threshold(y_val, p_val):
    cands = np.unique(np.quantile(p_val, np.linspace(0.05, 0.95, 19)))
    best_t, best_c = 0.5, np.inf
    for t in cands:
        c = contact_cost(y_val, (p_val >= t).astype(int))
        if c < best_c:
            best_c, best_t = c, float(t)
    return best_t

def metrics_dict(y, p, thr, k=None):
    y = np.asarray(y).astype(int); p = np.asarray(p, dtype=float)
    if k is None:
        k = max(1, int(K_FRACTION * len(y)))
    yhat = (p >= thr).astype(int)
    return {
        "PR-AUC": float(average_precision_score(y, p)),
        "precision@k": precision_at_k(y, p, k),
        "k": int(k),
        "Brier": float(brier_score_loss(y, p)),
        "kaštai": contact_cost(y, yhat),
        "slenkstis": float(thr),
        "n": int(len(y)),
        "positives": float(y.mean()),
    }

def plot_pr(curves, title, fname):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    plt.figure(figsize=(7,5))
    for name,(y,p) in curves.items():
        pr, rc, _ = precision_recall_curve(y, p)
        ap = average_precision_score(y, p)
        plt.plot(rc, pr, label=f"{name} (PR-AUC={ap:.3f})")
    plt.xlabel("Atgaminimas (Recall)"); plt.ylabel("Tikslumas (Precision)")
    plt.title(title); plt.legend(loc="lower left"); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, fname), dpi=140); plt.show()

def plot_cal(curves, title, fname):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    plt.figure(figsize=(7,5))
    for name,(y,p) in curves.items():
        frac, mp = calibration_curve(y, p, n_bins=10, strategy="quantile")
        plt.plot(mp, frac, marker="o", label=f"{name} (Brier={brier_score_loss(y,p):.3f})")
    plt.plot([0,1],[0,1],"k--", label="ideali kalibracija")
    plt.xlabel("Vidutinė p̂"); plt.ylabel("Stebėta teigiamų dalis")
    plt.title(title); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, fname), dpi=140); plt.show()

def save_preds(name, y, p):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    path = os.path.join(OUTPUT_DIR, f"preds_{name}.csv")
    pd.DataFrame({"y": y, "p": p}).to_csv(path, index=False)
    print("Išsaugota", path)

def load_other_preds(exclude):
    found = {}
    if not os.path.isdir(OUTPUT_DIR):
        return found
    for fn in sorted(os.listdir(OUTPUT_DIR)):
        if fn.startswith("preds_") and fn.endswith(".csv"):
            key = fn[len("preds_"):-4]
            if key == exclude:
                continue
            tab = pd.read_csv(os.path.join(OUTPUT_DIR, fn))
            if "y" in tab.columns and "p" in tab.columns:
                found[key] = (tab["y"].to_numpy(), tab["p"].to_numpy())
    return found

def show_probability_examples(te_df, y_true, p_hat, thr, n=10, model_name="model"):
    """
    Parodo, kaip p̂ naudojama praktiškai: operatorius skambina nuo didžiausios
    tikimybės. 10 test klientų + histograma visai test imčiai.
    „Teisus/klydo“ lyginama su kaštais parinktu slenksčiu thr (ne su 0,5),
    nes prie 11,7 % teigiamos klasės p̂ dažnai būna mažesnė nei 0,5.
    """
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    tab = te_df.copy()
    tab["p_hat"] = np.asarray(p_hat, dtype=float)
    tab["y_tikras"] = np.asarray(y_true).astype(int)
    tab["eilutes_nr"] = tab.index.astype(int)
    top = tab.sort_values("p_hat", ascending=False).head(n)
    cols = [
        c for c in [
            "eilutes_nr", "age", "job", "marital", "education", "balance",
            "housing", "loan", "contact", "month", "campaign", "poutcome",
            "never_contacted", "p_hat", "y_tikras",
        ]
        if c in top.columns
    ]
    print("10 test klientų, surikiuotų mažėjančia p̂ (kaip skambučių sąrašas):")
    display(top[cols].reset_index(drop=True))

    print("\nTas pats paprastais sakiniais:")
    for _, row in top.iterrows():
        p = float(row["p_hat"])
        tikras = "taip" if int(row["y_tikras"]) == 1 else "ne"
        pred_taip = p >= thr
        actual_taip = int(row["y_tikras"]) == 1
        verdiktas = "modelis teisus" if pred_taip == actual_taip else "modelis klydo"
        print(
            f"Klientas Nr. {int(row['eilutes_nr'])}: p̂={p:.2f} → {100*p:.0f}% "
            f"tikimybė sutikti, tikras atsakymas: {tikras} ({verdiktas})."
        )

    plt.figure(figsize=(7, 4))
    plt.hist(np.asarray(p_hat), bins=20, range=(0, 1), edgecolor="black", color="steelblue")
    plt.xlabel("p̂ (tikimybė, kad sutiks)")
    plt.ylabel("Kiek klientų")
    plt.title("Kaip pasiskirsto visos test imties tikimybės")
    plt.xlim(0, 1)
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    fname = os.path.join(OUTPUT_DIR, f"prob_hist_{model_name}.png")
    plt.savefig(fname, dpi=140)
    plt.show()
    print("Histograma išsaugota:", fname)


def prepare_parts(include_duration=False):
    raw = load_bank()
    print("y=yes dalis visame rinkinyje:", (raw["y"].str.lower()=="yes").mean())
    print("pdays=-1 dalis:", (raw["pdays"]==-1).mean(), "(dokumente ~81,7 %)")
    print("duration=0 eilučių:", (raw["duration"]==0).sum(), "(visos turėtų būti y=no — nutekėjimo požymis)")
    ch = month_change_indices(raw)
    print("Mėnesio periodų:", len(ch)-1)
    df = add_features(raw)
    tr, va, te = chronological_split(df)
    tr, va, te = cap_previous(tr, va, te)
    print(
        "y=1 dalis train/val/test:",
        round(tr["y_bin"].mean(), 4),
        round(va["y_bin"].mean(), 4),
        round(te["y_bin"].mean(), 4),
        "← vėlesnėse kampanijose sutarčių daugiau; todėl chronologija būtina.",
    )
    return tr, va, te


## 2. Duomenys

Tikriname eilučių skaičių ir tai, kad `pdays=-1` bei `duration=0` elgiasi kaip dokumente.

In [ ]:
tr, va, te = prepare_parts(include_duration=False)


## 3. Paruošimas CatBoost

CatBoost **moka skaityti kategorijas be one-hot** (job, month, poutcome...). Todėl čia nenaudojame OneHotEncoder pagrindiniam modeliui — paliekame tekstus. One-hot vis tiek naudosime logistiniam baseline hipotezės testui, kad palyginimas būtų sąžiningas su Baseline 2 notebook'u.

In [ ]:
MODEL_NAME = "catboost"
include_duration = False
X_tr_cb, y_tr = Xy(tr, include_duration)
X_va_cb, y_va = Xy(va, include_duration)
X_te_cb, y_te = Xy(te, include_duration)
cat_idx = [X_tr_cb.columns.get_loc(c) for c in CAT_COLS]
print("Kategoriniai indeksai CatBoost:", list(zip(CAT_COLS, cat_idx)))


## 4. Mokymas

`auto_class_weights='Balanced'` — analogas class_weight, nes teigiamų tik 11,7 %.

`eval_set` + `early_stopping` — sustojame, kai validation PR nebegerėja, kad nepersimokytume.

**Kitoje celėje prie `fit` paliktas komentaras su formule** $F_M(x)=\sum \gamma_m h_m(x)$, $\hat p=\sigma(F_M(x))$.

In [ ]:
model = CatBoostClassifier(
    iterations=400,
    depth=6,
    learning_rate=0.05,          # tai γ_m mastelis
    loss_function="Logloss",
    eval_metric="Logloss",
    random_seed=SEED,
    auto_class_weights="Balanced",
    verbose=100,
    od_type="Iter",
    od_wait=40,
    use_best_model=True,
)
# F_M(x) = Σ_m γ_m h_m(x);  p̂ = σ(F_M(x))
# fit() iš train duomenų išmoksta medžius h_m ir naudoja learning_rate kaip γ_m;
# predict_proba() pritaiko sigmoidę σ.
model.fit(X_tr_cb, y_tr, eval_set=(X_va_cb, y_va), cat_features=cat_idx)

p_va = model.predict_proba(X_va_cb)[:, 1]
p_te = model.predict_proba(X_te_cb)[:, 1]
thr = best_threshold(y_va, p_va)
print("Slenkstis (val, min. kaštai):", round(thr, 4))


## 4.1 Tikimybės paprastai

`predict_proba` čia ir yra $\hat p=\sigma(F_M(x))$. Rikiuojame test klientus nuo didžiausios tikimybės — taip bankas naudotų modelį ribotam skambučių biudžetui. Šalia rašome tikrą `y`, kad matytųsi, ar aukšta p̂ sutampa su „taip“.

In [ ]:
show_probability_examples(te, y_te, p_te, thr, n=10, model_name=MODEL_NAME)


## 5. Metrikos teste

In [ ]:
m = metrics_dict(y_te, p_te, thr)
display(pd.DataFrame([m], index=[MODEL_NAME]))
save_preds(MODEL_NAME, y_te, p_te)
plot_pr({MODEL_NAME: (y_te, p_te)}, "PR kreivė — CatBoost", "pr_catboost.png")
plot_cal({MODEL_NAME: (y_te, p_te)}, "Kalibracija — CatBoost", "calibration_catboost.png")


## 6. Hipotezė vs logistinė regresija

Kriterijus iš dokumento: CatBoost laimi, jei **PR-AUC skirtumas ≥ 0,03** ir **p < 0,05** (bootstrap, 1000 pakartojimų), abiem modeliams **be duration**, tame pačiame test rinkinyje.

Čia pačiame notebook'e apmokome tą patį Baseline 2, kad palyginimas veiktų net paleidus tik šį failą.

In [ ]:
# Tas pats one-hot pipeline kaip logreg notebook'e
prep_lr = make_preprocessor(False)
Xtr_lr = prep_lr.fit_transform(Xy(tr, False)[0])
Xva_lr = prep_lr.transform(Xy(va, False)[0])
Xte_lr = prep_lr.transform(Xy(te, False)[0])
lr = LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000, solver="lbfgs", random_state=SEED)
lr.fit(Xtr_lr, y_tr)
p_te_lr = lr.predict_proba(Xte_lr)[:, 1]

ap_cb = average_precision_score(y_te, p_te)
ap_lr = average_precision_score(y_te, p_te_lr)
print(f"CatBoost PR-AUC = {ap_cb:.4f}")
print(f"LogReg   PR-AUC = {ap_lr:.4f}")
print(f"Δ = {ap_cb-ap_lr:.4f}  (reikia ≥ 0.03)")

def bootstrap_diff(y, p1, p0, n_boot=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        diffs[i] = average_precision_score(y[idx], p1[idx]) - average_precision_score(y[idx], p0[idx])
    mean = float(diffs.mean())
    p = float(2 * min((diffs<=0).mean(), (diffs>=0).mean()))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return mean, min(p, 1.0), (float(lo), float(hi))

delta, pval, ci = bootstrap_diff(y_te, p_te, p_te_lr, n_boot=1000, seed=SEED)
print(f"Bootstrap Δ vidurkis = {delta:.4f}, 95% PI = [{ci[0]:.4f}, {ci[1]:.4f}], p = {pval:.4f}")
win = (delta >= 0.03) and (pval < 0.05)
print("HIPOTEZĖ PRIIMAMA." if win else "HIPOTEZĖ ATMETAMA.")
if win:
    print("CatBoost statistiškai reikšmingai lenkia tiesinį baseline: previous/poutcome/balance sąveikos ir kategorijų priklausomybės, kurių vienas linijinis balas nepagauna.")
else:
    print("Skirtumas per mažas arba nepatikimas. Tada pakanka paprastesnio logistinio modelio — neigiamas rezultatas dokumente irgi priimtinas.")
plot_pr({"CatBoost": (y_te, p_te), "LogReg": (y_te, p_te_lr)}, "CatBoost vs logistinė regresija", "pr_catboost_vs_logreg.png")
plot_cal({"CatBoost": (y_te, p_te), "LogReg": (y_te, p_te_lr)}, "Kalibracija: CatBoost vs LogReg", "cal_catboost_vs_logreg.png")


## 7. Abliacija (duration) ir laiko poslinkis

Abliacija — su/be `duration`.

Laiko poslinkis: CatBoost mokomas tik ant pirmų ~70 % **be** validation (kad vidurys nesimatytų mokant). Tada lyginame vidurį vs pabaigą.

In [ ]:
print("=== ABLIACIJA duration ===")
Xtr_d, _ = Xy(tr, True); Xva_d, _ = Xy(va, True); Xte_d, _ = Xy(te, True)
cat_idx_d = [Xtr_d.columns.get_loc(c) for c in CAT_COLS]
model_d = CatBoostClassifier(
    iterations=400, depth=6, learning_rate=0.05, loss_function="Logloss",
    eval_metric="Logloss", random_seed=SEED, auto_class_weights="Balanced",
    verbose=False, od_type="Iter", od_wait=40, use_best_model=True,
)
# F_M(x) = Σ_m γ_m h_m(x);  p̂ = σ(F_M(x))  — tas pats fit kaip pagrindiniame modelyje, tik su duration.
model_d.fit(Xtr_d, y_tr, eval_set=(Xva_d, y_va), cat_features=cat_idx_d)
p_te_d = model_d.predict_proba(Xte_d)[:, 1]
print(f"PR-AUC be duration = {ap_cb:.4f}")
print(f"PR-AUC su duration = {average_precision_score(y_te, p_te_d):.4f}")
print("Skirtumas parodo nutekėjimą: duration žinomas tik po skambučio.")
plot_pr({"be duration": (y_te, p_te), "su duration": (y_te, p_te_d)}, "Abliacija duration — CatBoost", "pr_ablation_catboost.png")

print("=== LAIKO POSLINKIS ===")
print("CatBoost čia mokomas TIK ant pirmų ~70 % (be validation), kad vidurys nebūtų matytas mokant.")
print("Tada tas pats modelis testuojamas: arti = vidurys, toli = pabaiga.")
model_s = CatBoostClassifier(
    iterations=400, depth=6, learning_rate=0.05, loss_function="Logloss",
    random_seed=SEED, auto_class_weights="Balanced", verbose=False,
)
# F_M(x) = Σ_m γ_m h_m(x);  p̂ = σ(F_M(x))
model_s.fit(X_tr_cb, y_tr, cat_features=cat_idx)
p_near = model_s.predict_proba(X_va_cb)[:, 1]
p_far = model_s.predict_proba(X_te_cb)[:, 1]
print(f"y=1 dažnis arti = {y_va.mean():.4f}  ({int(y_va.sum())}/{len(y_va)})")
print(f"y=1 dažnis toli = {y_te.mean():.4f}  ({int(y_te.sum())}/{len(y_te)})")
m_near = metrics_dict(y_va, p_near, 0.5)
m_far = metrics_dict(y_te, p_far, 0.5)
cmp = pd.DataFrame([m_near, m_far], index=["arti laike (vidurys)", "toli laike (pabaiga)"])
display(cmp[["n", "positives", "PR-AUC", "precision@k", "Brier", "kaštai"]])
print(f"PR-AUC (arti − toli) = {m_near['PR-AUC'] - m_far['PR-AUC']:.4f}")
print("Jei toli PR-AUC prastesnis — ankstyvų kampanijų taisyklės vėliau nebegalioja (concept drift).")
plot_pr(
    {"arti (vidurys)": (y_va, p_near), "toli (pabaiga)": (y_te, p_far)},
    "Laiko poslinkis — CatBoost",
    "pr_timeshift_catboost.png",
)


## 8. Klaidų analizė

Požymių svarba sako, **kuriais klausimais medžiai dalija klientus**. FN lentelė — konkretūs žmonės, kurie **sutiko**, bet modelis davė žemą $\hat p$ (praleisti pirkėjai; dokumente FN brangesnė klaida nei FP).

In [ ]:
imp = pd.DataFrame({
    "požymis": X_tr_cb.columns,
    "svarba": model.get_feature_importance(),
}).sort_values("svarba", ascending=False)
display(imp.head(15))
plt.figure(figsize=(7,5))
top = imp.head(12).iloc[::-1]
plt.barh(top["požymis"], top["svarba"])
plt.xlabel("CatBoost feature importance")
plt.title("Svarbiausi požymiai (be duration)")
plt.tight_layout()
os.makedirs(OUTPUT_DIR, exist_ok=True)
plt.savefig(os.path.join(OUTPUT_DIR, "feature_importance_catboost.png"), dpi=140)
plt.show()

# FN: tikras y=1, bet p̂ mažesnis už slenkstį (modelis neskambintų)
fn_mask = (y_te == 1) & (p_te < thr)
fn = te.loc[fn_mask].copy()
fn["p_hat"] = p_te[fn_mask]
fn = fn.sort_values("p_hat")  # labiausiai „netikėti“ sutikimai
cols_show = ["age","job","marital","education","balance","housing","loan","contact","month",
             "campaign","pdays","previous","poutcome","never_contacted","p_hat"]
print(f"FN skaičius teste (y=1 ir p̂ < {thr:.3f}): {len(fn)} iš {(y_te==1).sum()} teigiamų")
n_show = min(12, len(fn))
display(fn[cols_show].head(n_show))
fn[cols_show].head(n_show).to_csv(os.path.join(OUTPUT_DIR, "fn_examples_catboost.csv"), index=False)
print("Tipinės FN priežastys, kurių ieškoti lentelėje: poutcome=unknown (nėra sėkmės istorijos),")
print("mažas balance, housing=yes, contact=unknown, student/retired išimtys, kurias modelis nuvertina.")


## 9. Palyginimas su kitais notebook'ais

In [ ]:
# Palyginimas su kitais modeliais, jei jų notebook'ai jau paleisti (outputs/preds_*.csv)
others = load_other_preds(MODEL_NAME)
all_curves = {MODEL_NAME: (y_te, p_te), **others}
if len(all_curves) > 1:
    print("Rasti kiti modeliai:", list(others.keys()))
    rows = []
    for name,(yy,pp) in all_curves.items():
        # slenkstis 0.5 palyginimui tarp failų; tikrosios metrikos — kiekvieno notebook viduje
        rows.append({"modelis": name, **metrics_dict(yy, pp, 0.5)})
    display(pd.DataFrame(rows).set_index("modelis"))
    plot_pr(all_curves, "Precision–Recall: visi rasti modeliai", "pr_all_models.png")
    plot_cal(all_curves, "Kalibracija: visi rasti modeliai", "calibration_all_models.png")
else:
    print("Kitų modelių preds_*.csv nėra. Paleiskite kitus 3 notebook'us tame pačiame aplanke, tada perleiskite šią celę — atsiras bendri grafikai.")


Paleidus šią celę, į jūsų kompiuterio Atsisiuntimų (Downloads) aplanką atsisiųs ZIP failas su visais šio modelio rezultatais (preds_*.csv ir grafikais).

In [ ]:
import shutil
from google.colab import files
shutil.make_archive(f"{MODEL_NAME}_outputs", "zip", "outputs")
files.download(f"{MODEL_NAME}_outputs.zip")
